<a href="https://colab.research.google.com/github/teejx/CCRNFLRL_EXERCISE/blob/main/Exercise2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 2

In [900]:
from math import sqrt, log
import pandas as pd

In [901]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 1: Enter your data

Add a new row after each pull: Pull number, Machine played ("A", "B", "C"), and Reward earned.

In [902]:
data = pd.DataFrame({
    "Pull": [1, 2, 3],          # Pull number
    "Machine": ["A", "B", "C"], # Which machine was played
    "Reward": [7, 8, 5]         # Reward received from that machine
})

In [903]:
# To load data from a Google Sheet, you need the SHEET_ID and SHEET_NAME.
# You can find the SHEET_ID in the URL of your Google Sheet, between "/d/" and "/edit".
# The SHEET_NAME is usually "Sheet1" by default, but you should verify the name of the sheet containing your data.
# Ensure your Google Sheet is shared with "Anyone with the link" as Viewer.

SHEET_ID = '111dwne353NAMAJli4oTVf_1KT9VT0HT-BZFRiENVOwo'
SHEET_NAME = 'UCB' # Replace with your actual Google Sheet name
url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/gviz/tq?tqx=out:csv&sheet={SHEET_NAME}'

try:
    data = pd.read_csv(url)
    print("Data loaded successfully:")
    display(data)
except Exception as e:
    print(f"Error loading data: {e}")
    print("Please ensure the sheet is publicly accessible or shared with 'Anyone with the link' as Viewer.")

Data loaded successfully:


,Pull #,Machine,Reward,UCB Score A,UCB Score B,UCB Score C,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,1,A,0.0,1.4823,1.4823,1.4823,NaN,NaN,CANDY
1,2,B,0.0,1.4823,1.4823,1.4823,NaN,NaN,PARTY
2,3,C,0.0,1.4823,1.4823,1.4823,NaN,NaN,FRUIT
3,4,A,0.0,1.1774,1.6651,1.6651,NaN,NaN,NaN
4,5,B,0.0,1.2686,1.2686,1.7941,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
295,296,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
296,297,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
297,298,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
298,299,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Step 2: Calculate UCB

In [904]:
def calculate_ucb(df):
    results = []
    total_pulls = df.dropna(subset=['Machine'])['Pull #'].max()
    print(total_pulls)
    if pd.isna(total_pulls):
        total_pulls = 0

    machines = sorted(df["Machine"].dropna().unique())

    for machine in machines:
        machine_data = df[df["Machine"] == machine].dropna(subset=['Pull #', 'Reward'])
        n_j = len(machine_data)
        avg_reward = machine_data["Reward"].mean() if n_j > 0 else 0
        exploration_bonus = sqrt((2 * log(total_pulls)) / n_j) if n_j > 0 and total_pulls > 0 else float("inf")
        ucb_score = avg_reward + exploration_bonus
        results.append({
            "Machine": machine,
            "Pulls": n_j,
            "Avg Reward": round(avg_reward, 2),
            "Exploration Bonus": round(exploration_bonus, 4),
            "UCB Score": round(ucb_score, 4)
        })
    return pd.DataFrame(results).sort_values(by="UCB Score", ascending=False)


# Step 3: Calculate and display


In [905]:
ucb_table = calculate_ucb(data)
print("📊 UCB Results Table")
print(ucb_table)

100
📊 UCB Results Table
  Machine  Pulls  Avg Reward  Exploration Bonus  UCB Score
0       A     71        2.85             0.3602     3.2052
2       C      6        0.00             1.2390     1.2390
1       B     23        0.57             0.6328     1.1980


In [906]:
last_input_index = data.dropna(subset=['Machine']).index.max()

if pd.isna(last_input_index):
    print("No machine data has been inputted yet to update UCB scores.")
else:

    ucb_a_score = ucb_table[ucb_table['Machine'] == 'A']['UCB Score'].iloc[0] if 'A' in ucb_table['Machine'].values else None
    ucb_b_score = ucb_table[ucb_table['Machine'] == 'B']['UCB Score'].iloc[0] if 'B' in ucb_table['Machine'].values else None
    ucb_c_score = ucb_table[ucb_table['Machine'] == 'C']['UCB Score'].iloc[0] if 'C' in ucb_table['Machine'].values else None

    # Update the corresponding UCB score columns in the last inputted row
    if 'UCB Score A' in data.columns and ucb_a_score is not None:
        data.loc[last_input_index, 'UCB Score A'] = ucb_a_score
    else:
         print("Warning: Could not update 'UCB Score A'. Column not found or Machine A score not available.")

    if 'UCB Score B' in data.columns and ucb_b_score is not None:
        data.loc[last_input_index, 'UCB Score B'] = ucb_b_score
    else:
        print("Warning: Could not update 'UCB Score B'. Column not found or Machine B score not available.")

    if 'UCB Score C' in data.columns and ucb_c_score is not None:
        data.loc[last_input_index, 'UCB Score C'] = ucb_c_score
    else:
        print("Warning: Could not update 'UCB Score C'. Column not found or Machine C score not available.")

    print(ucb_a_score)
    print(ucb_b_score)
    print(ucb_c_score)

    # Calculate and display average Reward scores
    avg_reward_a = data[data['Machine'] == 'A']['Reward'].mean()
    avg_reward_b = data[data['Machine'] == 'B']['Reward'].mean()
    avg_reward_c = data[data['Machine'] == 'C']['Reward'].mean()


    print(f"\nAverage Reward A: {avg_reward_a:.4f}")
    print(f"Average Reward B: {avg_reward_b:.4f}")
    print(f"Average Reward C: {avg_reward_c:.4f}")


    print(f"\nUCB scores for Machines A, B, and C updated in the last inputted row (index: {last_input_index}) of the data DataFrame.")
    # Display the updated last row with the relevant columns
    display(data.loc[[last_input_index], ['Pull #', 'Machine', 'Reward', 'UCB Score A', 'UCB Score B', 'UCB Score C']])

3.2052
1.198
1.239

Average Reward A: 2.8451
Average Reward B: 0.5652
Average Reward C: 0.0000

UCB scores for Machines A, B, and C updated in the last inputted row (index: 99) of the data DataFrame.


,Pull #,Machine,Reward,UCB Score A,UCB Score B,UCB Score C
99,100,A,10.0,3.2052,1.198,1.239


# After 100 pulls we found that **slot machine A** has the most winnings with the average reward of 2.8451 while **slot machine B** has the average of 0.5652 while at **slot machine C** we won nothing with an average score of 0.0000

In [907]:
best_machine = ucb_table.iloc[0]["Machine"]
print(f"\n🎯 Recommended next machine to play: {best_machine}")


🎯 Recommended next machine to play: A


In [908]:
output_columns = ['Pull #', 'Machine', 'Reward', 'UCB Score A', 'UCB Score B', 'UCB Score C']

if 'UCB Score A' not in data.columns:
    print("Error: 'UCB Score' column not found in the DataFrame. Please ensure the UCB calculation and merge steps ran successfully.")

else:
    output_df = data[output_columns]

    output_filename = 'ucb_results.csv'
    drive_path = '/content/drive/MyDrive/'

    full_output_path = drive_path + output_filename

    output_df.to_csv(full_output_path, index=False)

    print(f"Data saved to '{full_output_path}' with columns: {output_columns}")

Data saved to '/content/drive/MyDrive/ucb_results.csv' with columns: ['Pull #', 'Machine', 'Reward', 'UCB Score A', 'UCB Score B', 'UCB Score C']
